# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a scientific dataset using the `mlcroissant` library. We focus on record sets and fields by referencing their `@id` values, ensuring semantic clarity and reproducibility.

### Dataset Source
The dataset is described in Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not available
!pip install mlcroissant

## 1. Data Loading
We begin by using `mlcroissant` to load the dataset's metadata and record structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing metadata attributes
md = dataset.metadata  # Note: No subscripting, treat as object
print(f"{md.name}: {md.description}")
print(f"Version: {md.version}\nLicense: {md.license}")

## 2. Data Overview
Reviewing all available record sets and their fields, with proper `@id` usage.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print("Available RecordSets and their fields:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {rs.name}")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for further analysis. All references are made using the respective `@id`.

In [ ]:
# Prepare list of RecordSet @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each RecordSet into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded RecordSet @id: {rs_id} with {len(df)} rows and columns: {df.columns.tolist()}")

# Example: show columns and first rows of the first RecordSet
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {example_rs_id}: {dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We will select one RecordSet, then pick a numeric Field (referenced by its `@id`), and demonstrate filtering, normalization, and grouping operations using Pandas.

In [ ]:
# For demonstration, use the first available RecordSet
selected_rs_id = record_set_ids[0]
df = dataframes[selected_rs_id]

# Try to automatically identify a numeric field by data type
selected_field_id = None
for field in dataset.metadata.record_set(selected_rs_id).fields:
    if field.data_type in ["Float", "Integer", "Number"]:
        selected_field_id = field.id
        break

# Use the first found numeric field
if selected_field_id is not None and selected_field_id in df.columns:
    print(f"Selected numeric field for analysis: {selected_field_id}")
    # Example threshold
    threshold = df[selected_field_id].dropna().mean() if not df[selected_field_id].dropna().empty else 0
    threshold = threshold if pd.notnull(threshold) else 0
    filtered_df = df[df[selected_field_id] > threshold]
    print(f"Filtered records with {selected_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    # Z-normalization
    filtered_df[f"{selected_field_id}_normalized"] = (filtered_df[selected_field_id] - filtered_df[selected_field_id].mean()) / filtered_df[selected_field_id].std()
    display(filtered_df[[selected_field_id, f"{selected_field_id}_normalized"]].head())
    # Attempt to group by a categorical field
    group_field_id = None
    for field in dataset.metadata.record_set(selected_rs_id).fields:
        if field.data_type in ["Text"] and field.id in df.columns:
            group_field_id = field.id
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[selected_field_id].mean()
        print(f"Mean of {selected_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis in this RecordSet.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if available, its relationship to a categorical field using Matplotlib and Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_field_id and selected_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[selected_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {selected_field_id} in {selected_rs_id}")
    plt.xlabel(selected_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=selected_field_id, data=df)
        plt.title(f"{selected_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We loaded and explored mass spectrometry protein data using the `mlcroissant` library, referencing all entities by their `@id`.
- We reviewed available record sets, fields, and example records.
- We performed basic filtering, normalization, grouping, and visualization to gain insights into the dataset.

For further analysis, consult the Croissant schema and metadata for specific meanings of each field and data curation protocols.
<br><br>
_Notebook generated with FAIR principles using Croissant metadata schema._